# 작은 TCN과 고정 높이 규칙

학습 선수 한 명을 매번 제외합니다. 수동 ready~drive 구간 안에서 무릎 통과 후보를 골라 두 판독 중 가까운 경계에서 벗어난 시간을 비교합니다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").exists())
sys.path.insert(0, str(root))
import pandas as pd
import matplotlib.pyplot as plt
from wlcv.evaluation.knee_pass_tcn import run
from wlcv.evaluation.train_knee_pass_coverage import PARENT, extract

저장된 점수를 먼저 확인합니다. 다시 학습하려면 retrain을 True로 바꾸세요. 59번 실행의 좌표와 수동 판독을 사용합니다.

In [ ]:
retrain = False
parent = root / "outputs/knee_pass_tcn"
runs = sorted(parent.glob("*/scores.csv"))
if retrain or not runs:
    sources = sorted(PARENT.glob("*/frames.csv"))
    source = sources[-1].parent if sources else extract()
    folder, scores = run(source)
else:
    folder = runs[-1].parent
    scores = pd.read_csv(runs[-1])
print(folder)

In [ ]:
main = scores[scores.seed.fillna(0).eq(0)]
table = main.pivot(index="video",columns="method",values="outside_ms")
display(table.round(1))
common = table.dropna().rename(columns={"높이 규칙":"Height rule", "시간 중앙":"Time midpoint"})
ax = common.abs().plot.bar(figsize=(8,4))
ax.set(xlabel="Held-out athlete / video", ylabel="Distance outside manual interval (ms)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

같은 4편에서 기준 TCN의 평균 절대 이탈 시간은 254.6 ms, 높이 규칙은 64.6 ms였습니다. 보류한 3편은 표에 빈칸으로 표시됩니다.